<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">

# The Data Scientist
## Book 3 · Statistics, Machine Learning, and Model Evaluation
### Notebook 06 · Module 3 Evaluation Report
© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br> The Python Quants GmbH | https://tpq.io<br>

https://thedatascientist.dev | https://linktr.ee/dyjh

### Why this notebook exists
The chapter asks for an evaluation report, not a model dump. The notebook should read
like a small project write-up that a colleague could follow.

## Setup
We make the notebook portable first so the reporting helper and dataset load
from a stable book root.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "3_data"
REPO_URL = "https://github.com/yhilpisch/tdscode"

cwd = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in (cwd, *cwd.parents):
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")
    if not repo_root.exists():
        # Clone the companion repo once in Colab.
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
    BOOK_ROOT = repo_root / BOOK_NAME

os.chdir(BOOK_ROOT)

# Make the book root and the helper folder importable.
for path in (BOOK_ROOT, BOOK_ROOT / "code"):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

requirements = BOOK_ROOT / "requirements.txt"
if requirements.exists() and "google.colab" in sys.modules:
    # Keep Colab aligned with the book dependencies.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

print("Book root:", BOOK_ROOT)


## Load the Reporting Helper
The helper module keeps the reusable churn-report logic outside the notebook so
the report can stay readable.


In [ ]:
from pathlib import Path
import importlib.util

import pandas as pd

module_path = BOOK_ROOT / 'code' / 'churn_report.py'
spec = importlib.util.spec_from_file_location('churn_report', module_path)
assert spec is not None and spec.loader is not None
churn_report = importlib.util.module_from_spec(spec)
spec.loader.exec_module(churn_report)

print(f'Loaded {module_path.resolve()}')


## Inspect the Project Data First
An evaluation report should begin with the data, not with the model. The next
cell prints the head, class balance, and numeric summary before any fitting.


In [ ]:
df = churn_report.load_customers()

print(df.head().to_string(index=False))
print()
print('churn counts:')
print(df[churn_report.TARGET_COLUMN].value_counts().sort_index())
print()
print('numeric summary:')
print(df[churn_report.NUMERIC_COLUMNS].describe().round(2).to_string())


## Compare Candidate Models
Once the data is visible, we fit several models and collect validation and test
metrics in one table. That is the core evidence the report needs.


In [ ]:
X_train, X_valid, X_test, y_train, y_valid, y_test = churn_report.split_customers(df)
models = churn_report.build_models()

rows = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    valid_pred = model.predict(X_valid)
    test_pred = model.predict(X_test)

    valid_scores = churn_report.evaluate_predictions(y_valid, valid_pred)
    test_scores = churn_report.evaluate_predictions(y_test, test_pred)

    rows.append({'model': name, 'split': 'validation', **valid_scores})
    rows.append({'model': name, 'split': 'test', **test_scores})

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))


## Choose the Validation Winner
The final code step names the model that wins on the validation split so the
plain-language summary that follows has a concrete choice to justify.


In [ ]:
validation_rows = comparison[comparison['split'] == 'validation'].copy()
validation_sorted = validation_rows.sort_values(
    ['recall', 'accuracy'], ascending=False
)
best_name = validation_sorted.iloc[0]['model']
print('best validation model =', best_name)
best_model = fitted_models[best_name]


### Report Summary
The model choice should be explained in plain language. A good summary says what was
tried, which model was chosen, what it still gets wrong, and what data would help
next.

### Where We Are Heading Next
The module ends here. The next step in a real project would be to package the
notebook, the data, and the short README into a small public repo.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">